# Parte 1: Model Checking Proposicional

En esta parte implementarás las funciones de **model checking** para lógica proposicional.

## Objetivos
1. Generar todos los modelos posibles para un conjunto de átomos
2. Verificar satisfacibilidad de fórmulas
3. Verificar si una fórmula es tautología
4. Verificar si una base de conocimiento implica una consulta
5. Generar tablas de verdad

In [ ]:
import sys
sys.path.insert(0, '..')

from src.logic_core import Atom, Not, And, Or, Implies, Iff, evaluate, get_atoms
from src.utils import formula_to_string, print_truth_table

## Paso 0: Familiarízate con el AST

Antes de implementar, experimenta con las clases de `logic_core.py`.

In [ ]:
# Crear fórmulas
p = Atom('p')
q = Atom('q')

# p → q
f1 = Implies(p, q)
print(f"Fórmula: {formula_to_string(f1)}")
print(f"Átomos: {get_atoms(f1)}")
print(f"Evaluación con p=True, q=False: {evaluate(f1, {'p': True, 'q': False})}")
print(f"Evaluación con p=True, q=True: {evaluate(f1, {'p': True, 'q': True})}")

In [ ]:
# Fórmula más compleja: (p ∧ q) → (p ∨ r)
r = Atom('r')
f2 = Implies(And(p, q), Or(p, r))
print(f"Fórmula: {formula_to_string(f2)}")
print(f"Átomos: {get_atoms(f2)}")

### Bicondicional (↔)

El bicondicional `Iff(p, q)` es verdadero cuando ambos tienen el mismo valor de verdad: ambos verdaderos o ambos falsos. Se lee "p si y solo si q".

In [ ]:
# Bicondicional: p ↔ q
f3 = Iff(p, q)
print(f"Fórmula: {formula_to_string(f3)}")
print(f"p=True,  q=True:  {evaluate(f3, {'p': True, 'q': True})}")
print(f"p=True,  q=False: {evaluate(f3, {'p': True, 'q': False})}")
print(f"p=False, q=True:  {evaluate(f3, {'p': False, 'q': True})}")
print(f"p=False, q=False: {evaluate(f3, {'p': False, 'q': False})}")

# Tabla de verdad del bicondicional
print("\nTabla de verdad de p ↔ q:")
print_truth_table(f3)

## Paso 1: Implementar `get_all_models`

Abre `src/model_checking.py` e implementa la función `get_all_models`.

**Hint:** Para n átomos, hay 2^n modelos posibles. Piensa en cómo los números del 0 al 2^n - 1 en binario te dan todas las combinaciones de True/False.

In [ ]:
from src.model_checking import get_all_models

# Prueba tu implementación
models = get_all_models({'p', 'q'})
print(f"Número de modelos: {len(models)}")
for m in models:
    print(f"  {m}")

# Debe imprimir 4 modelos con todas las combinaciones

## Paso 2: Implementar `check_satisfiable`

Una fórmula es **satisfacible** si existe al menos un modelo donde es verdadera.

**Hint:** Usa `get_all_models` y `evaluate`.

In [ ]:
from src.model_checking import check_satisfiable

# p ∧ q es satisfacible
sat, model = check_satisfiable(And(Atom('p'), Atom('q')))
print(f"p ∧ q es satisfacible: {sat}, modelo: {model}")

# p ∧ ¬p es insatisfacible
sat, model = check_satisfiable(And(Atom('p'), Not(Atom('p'))))
print(f"p ∧ ¬p es satisfacible: {sat}, modelo: {model}")

## Paso 3: Implementar `check_valid`

Una fórmula es **válida** (tautología) si es verdadera en todos los modelos.

**Hint:** φ es válida ⟺ ¬φ es insatisfacible.

In [ ]:
from src.model_checking import check_valid

# p ∨ ¬p es tautología
print(f"p ∨ ¬p es válida: {check_valid(Or(Atom('p'), Not(Atom('p'))))}")

# p → p es tautología
print(f"p → p es válida: {check_valid(Implies(Atom('p'), Atom('p')))}")

# p no es tautología
print(f"p es válida: {check_valid(Atom('p'))}")

## Paso 4: Implementar `check_entailment`

KB |= q si en todo modelo donde la KB es verdadera, q también lo es.

**Hint:** KB |= q ⟺ KB ∧ ¬q es insatisfacible.

In [ ]:
from src.model_checking import check_entailment

# Modus ponens: {p → q, p} |= q
kb = [Implies(Atom('p'), Atom('q')), Atom('p')]
print(f"Modus ponens: {check_entailment(kb, Atom('q'))}")

# No se puede derivar q solo de p → q
kb = [Implies(Atom('p'), Atom('q'))]
print(f"Solo implicación: {check_entailment(kb, Atom('q'))}")

## Paso 5: Implementar `truth_table`

In [ ]:
from src.model_checking import truth_table

# Tabla de verdad de p → q
table = truth_table(Implies(Atom('p'), Atom('q')))
for model, result in table:
    print(f"  {model} → {result}")

### Visualizar tabla de verdad con `print_truth_table`

La función `print_truth_table` imprime la tabla de verdad como una tabla ASCII formateada.

In [ ]:
# Tabla de verdad de p → q (formateada)
print_truth_table(Implies(Atom('p'), Atom('q')))

print()

# Tabla con 3 átomos: (p ∧ q) → (p ∨ r)
print_truth_table(Implies(And(Atom('p'), Atom('q')), Or(Atom('p'), Atom('r'))))

## Aplicación: El Caso Criminal

Ahora aplica tu model checking al caso criminal.

In [ ]:
# Base de conocimiento del caso criminal
kb_criminal = [
    Or(Atom('mayordomo_en_cocina'), Atom('mayordomo_en_biblioteca')),
    Implies(Atom('mayordomo_en_cocina'), Not(Atom('envenenó_copa'))),
    Implies(Not(Atom('llovía')), Not(Atom('jardín_mojado'))),
    Atom('jardín_mojado'),
    Or(Atom('envenenó_copa'), Atom('cocinera_envenenó')),
    Implies(Atom('cocinera_envenenó'), Atom('acceso_arsénico')),
    Not(Atom('acceso_arsénico')),
]

# Pregunta 1: ¿Llovía esa noche?
print(f"¿Llovía? {check_entailment(kb_criminal, Atom('llovía'))}")

# Pregunta 2: ¿La cocinera envenenó la copa?
print(f"¿Cocinera envenenó? {check_entailment(kb_criminal, Atom('cocinera_envenenó'))}")
print(f"¿Cocinera NO envenenó? {check_entailment(kb_criminal, Not(Atom('cocinera_envenenó')))}")

# Pregunta 3: ¿El mayordomo envenenó la copa?
print(f"¿Mayordomo envenenó? {check_entailment(kb_criminal, Atom('envenenó_copa'))}")

## Verificar con tests

Ejecuta los tests para validar tu implementación:

In [ ]:
!cd .. && python -m pytest tests/test_model_checking.py -v